In [1]:
from enum import Enum
from pathlib import Path
from typing import Tuple

import numpy as np

import itertools
import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from embed_data import EmbeddingTrainingMode, HypersphereEmbeddedDynamicsDataset

from model import EmbeddedDynamicsModel

In [2]:
torch.set_default_dtype(torch.float64)
device_cuda = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
device_cuda

device(type='cuda')

In [3]:
HYPERSPHERE_DIM = 1
HYPERSPHERE_RADII = [1.0]  # , 0.5, 2.0]  for now

TRAINING_EPOCHS = 1000  # matches nonembedded
TRAINING_LR = [1e-4]  # matches nonembedded
TRAINING_BATCH_SIZE = [128]  # matches nonembedded

TRAINING_DATA_MODES = [
    # use_intrinsic_coords, mode
    (False, EmbeddingTrainingMode.TRAIN_NO_ENC_DEC_STREAM),  # baseline to compare against
    (True, EmbeddingTrainingMode.TRAIN_NO_ENC_DEC_STREAM),
    (True, EmbeddingTrainingMode.TRAIN_EXTRINSIC_ENC_DEC_STREAM),
    (True, EmbeddingTrainingMode.TRAIN_WITH_VEL_ENC_STREAM),
    (True, EmbeddingTrainingMode.TRAIN_WOUT_VEL_ENC_STREAM)
]

MODEL_PARAMS = [
    # dec_num_layers, dec_num_nodes, mlp_num_layers, mlp_num_nodes, enc_num_layers, enc_num_nodes
    (4, 60, 4, 60, 4, 30)
]
PENALTY_PARAMS = [0.000, 0.001, 0.01]
USE_PREV_STATES = [True, False]

ACTIVATION_FNS = [nn.LeakyReLU]

DYNAMICS_DATA_DIR = Path("../../../data/unforced_dynamics")
TRAINING_RESULTS_DIR = Path("../../../data/training")

DYNAMICS_SUBDIR_FORMAT = "dim_{dim}/radius_{radius}"
TRAINING_SUBDIR_FORMAT = "dim_{dim}/radius_{radius}"

In [4]:
def train_loop(
        dataloader: DataLoader,
        model: EmbeddedDynamicsModel,
        loss_fn: nn.Module,
        optimizer: torch.optim.Optimizer,
        training_mode: EmbeddingTrainingMode,
        sparsity_penalty: float,
        use_intrinsic_pipeline_inputs: bool = True
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    model.train()

    batch_pipeline_loss = torch.zeros(len(dataloader), device=device_cpu)
    batch_dec_enc_loss = torch.zeros(len(dataloader), device=device_cpu)
    batch_sparsity_loss = torch.zeros(len(dataloader), device=device_cpu)

    for batch_idx, (prev_intrin_pos, prev_intrin_vel,
                    curr_intrin_pos, curr_intrin_vel,
                    next_intrin_pos, next_intrin_vel,

                    prev_extrin_pos, prev_extrin_vel,
                    curr_extrin_pos, curr_extrin_vel,
                    next_extrin_pos, next_extrin_vel) in enumerate(dataloader):

        # # copy the current batch to the gpu
        prev_intrin_pos = prev_intrin_pos.to(device_cuda, non_blocking=True)
        prev_intrin_vel = prev_intrin_vel.to(device_cuda, non_blocking=True)
        curr_intrin_pos = curr_intrin_pos.to(device_cuda, non_blocking=True)
        curr_intrin_vel = curr_intrin_vel.to(device_cuda, non_blocking=True)
        next_intrin_pos = next_intrin_pos.to(device_cuda, non_blocking=True)
        # next_intrin_vel = next_intrin_vel.to(device_cuda, non_blocking=True)

        prev_extrin_pos = prev_extrin_pos.to(device_cuda, non_blocking=True)
        prev_extrin_vel = prev_extrin_vel.to(device_cuda, non_blocking=True)
        curr_extrin_pos = curr_extrin_pos.to(device_cuda, non_blocking=True)
        curr_extrin_vel = curr_extrin_vel.to(device_cuda, non_blocking=True)
        next_extrin_pos = next_extrin_pos.to(device_cuda, non_blocking=True)

        total_loss = torch.tensor(0.0, device=device_cuda)

        # note that in all usual cases we will be passing intrinsic coordinates into the network but we need a best case baseline we
        # need to compare against which would correspond to passing in the extrinsic coordinates
        # print(f"use_intrinsic_pipeline_inputs: {use_intrinsic_pipeline_inputs}")
        if use_intrinsic_pipeline_inputs:
            # computes the loss directly associated with the final predicted intrinsic position
            if model.with_prev_state:
                pred_next_intrin_pos = model(curr_intrin_pos, curr_intrin_vel,
                                             prev_intrin_pos, prev_intrin_vel)
            else:
                pred_next_intrin_pos = model(curr_intrin_pos, curr_intrin_vel)
            pred_next_intrin_pos_loss = loss_fn(pred_next_intrin_pos, next_intrin_pos)

            total_loss += pred_next_intrin_pos_loss
            batch_pipeline_loss[batch_idx] = pred_next_intrin_pos_loss.item()
        else:
            # computes the loss directly associated with the final predicted extrinsic position
            if model.with_prev_state:
                pred_next_extrin_pos = model(curr_extrin_pos, curr_extrin_vel,
                                             prev_extrin_pos, prev_extrin_vel)
            else:
                pred_next_extrin_pos = model(curr_extrin_pos, curr_extrin_vel)
            pred_next_extrin_pos_loss = loss_fn(pred_next_extrin_pos, next_extrin_pos)

            total_loss += pred_next_extrin_pos_loss
            batch_pipeline_loss[batch_idx] = pred_next_extrin_pos_loss.item()

        # print(f"finished initial inference")

        # computes the loss associated with the decoder-encoder stream
        if training_mode == EmbeddingTrainingMode.TRAIN_NO_ENC_DEC_STREAM:
            pred_dec_enc_loss = torch.zeros(())  # this loss is not calculated
        elif training_mode == EmbeddingTrainingMode.TRAIN_EXTRINSIC_ENC_DEC_STREAM:
            pred_curr_extrin_pos, pred_curr_extrin_vel = model.get_extrinsic_from_decoder(curr_intrin_pos,
                                                                                          curr_intrin_vel)
            pred_curr_intrin_pos = model.get_intrinsic_pos_from_encoder(curr_extrin_pos)

            pred_dec_loss = (loss_fn(pred_curr_extrin_pos, curr_extrin_pos)
                             + loss_fn(pred_curr_extrin_vel, curr_extrin_vel))
            pred_enc_loss = loss_fn(pred_curr_intrin_pos, curr_intrin_pos)

            pred_dec_enc_loss = pred_dec_loss + pred_enc_loss
        elif training_mode == EmbeddingTrainingMode.TRAIN_WITH_VEL_ENC_STREAM:
            pred_curr_intrin_pos, pred_curr_intrin_vel = model.get_intrinsic_pos_vel_from_dec_enc(curr_intrin_pos,
                                                                                                  curr_intrin_vel)
            pred_dec_enc_loss = (loss_fn(pred_curr_intrin_pos, curr_intrin_pos)
                                 + loss_fn(pred_curr_intrin_vel, curr_intrin_vel))
        elif training_mode == EmbeddingTrainingMode.TRAIN_WOUT_VEL_ENC_STREAM:
            pred_curr_intrin_pos = model.get_intrinsic_pos_from_dec_enc(curr_intrin_pos, curr_intrin_vel)
            pred_dec_enc_loss = loss_fn(pred_curr_intrin_pos, curr_intrin_pos)
        else:
            raise NotImplementedError  # should never reach here

        total_loss += pred_dec_enc_loss
        batch_dec_enc_loss[batch_idx] = pred_dec_enc_loss.item()

        # computes a penalty on sparsity of the dynamic MLP
        sparsity_loss = sparsity_penalty * sum(torch.sum(param.flatten()) for param in model.get_dyn_mlp_params())
        batch_sparsity_loss[batch_idx] = sparsity_loss.item()
        total_loss += sparsity_loss

        # performs backpropagation
        total_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return batch_pipeline_loss, batch_dec_enc_loss, batch_sparsity_loss


def valid_loop(
        dataloader: DataLoader,
        model: EmbeddedDynamicsModel,
        loss_fn: nn.Module,
        use_intrinsic_pipeline_inputs: bool = True
) -> torch.Tensor:
    model.eval()

    # print(f"dataloader len: {len(dataloader)}")

    batch_pipeline_loss = torch.zeros(len(dataloader), device=device_cpu)

    # print(f"batch_pipeline_loss: {batch_pipeline_loss.shape}")

    for batch_idx, (prev_intrin_pos, prev_intrin_vel,
                    curr_intrin_pos, curr_intrin_vel,
                    next_intrin_pos, next_intrin_vel,

                    prev_extrin_pos, prev_extrin_vel,
                    curr_extrin_pos, curr_extrin_vel,
                    next_extrin_pos, next_extrin_vel) in enumerate(dataloader):

        # # copy the current batch to the gpu
        prev_intrin_pos = prev_intrin_pos.to(device_cuda, non_blocking=True)
        prev_intrin_vel = prev_intrin_vel.to(device_cuda, non_blocking=True)
        curr_intrin_pos = curr_intrin_pos.to(device_cuda, non_blocking=True)
        curr_intrin_vel = curr_intrin_vel.to(device_cuda, non_blocking=True)
        next_intrin_pos = next_intrin_pos.to(device_cuda, non_blocking=True)
        # next_intrin_vel = next_intrin_vel.to(device_cuda, non_blocking=True)

        prev_extrin_pos = prev_extrin_pos.to(device_cuda, non_blocking=True)
        prev_extrin_vel = prev_extrin_vel.to(device_cuda, non_blocking=True)
        curr_extrin_pos = curr_extrin_pos.to(device_cuda, non_blocking=True)
        curr_extrin_vel = curr_extrin_vel.to(device_cuda, non_blocking=True)
        next_extrin_pos = next_extrin_pos.to(device_cuda, non_blocking=True)

        # note that in all usual cases we will be passing intrinsic coordinates into the network but we need a best case baseline we
        # need to compare against which would correspond to passing in the extrinsic coordinates
        if use_intrinsic_pipeline_inputs:
            # computes the loss directly associated with the final predicted intrinsic position
            if model.with_prev_state:
                pred_next_intrin_pos = model(curr_intrin_pos, curr_intrin_vel,
                                             prev_intrin_pos, prev_intrin_vel)
            else:
                pred_next_intrin_pos = model(curr_intrin_pos, curr_intrin_vel)
            pred_next_intrin_pos_loss = loss_fn(pred_next_intrin_pos, next_intrin_pos)

            # print(f"pred_next_intrin_pos_loss: {pred_next_intrin_pos_loss}")
            # print(f"batch_pipeline_loss: {batch_pipeline_loss.shape}")

            batch_pipeline_loss[batch_idx] = pred_next_intrin_pos_loss.item()
        else:
            # computes the loss directly associated with the final predicted extrinsic position
            # print(f"curr_extrin_pos: {curr_extrin_pos.shape}")
            # print(f"curr_extrin_vel: {curr_extrin_vel.shape}")

            if model.with_prev_state:
                pred_next_extrin_pos = model(curr_extrin_pos, curr_extrin_vel,
                                             prev_extrin_pos, prev_extrin_vel)
            else:
                pred_next_extrin_pos = model(curr_extrin_pos, curr_extrin_vel)
            pred_next_extrin_pos_loss = loss_fn(pred_next_extrin_pos, next_extrin_pos)

            # print(f"pred_next_extrin_pos_loss: {pred_next_extrin_pos_loss}")
            # print(f"batch_pipeline_loss: {batch_pipeline_loss.shape}")

            batch_pipeline_loss[batch_idx] = pred_next_extrin_pos_loss.item()

    return batch_pipeline_loss


In [5]:
cfgs = list(itertools.product(
    HYPERSPHERE_RADII, MODEL_PARAMS,
    TRAINING_LR, TRAINING_BATCH_SIZE, PENALTY_PARAMS, USE_PREV_STATES,
    ACTIVATION_FNS, TRAINING_DATA_MODES
))
RESUME_SKIP = 5

for i, (radius,
        (dec_num_layers, dec_num_nodes,
         mlp_num_layers, mlp_num_nodes,
         enc_num_layers, enc_num_nodes),
        lr, batch_size, penalty, use_prev_state, activation_fn,
        (use_intrinsic, mode),) in enumerate(cfgs[RESUME_SKIP:]):
    print(f"cfg: {i+RESUME_SKIP}/{len(cfgs)}")

    training_dir = DYNAMICS_DATA_DIR / DYNAMICS_SUBDIR_FORMAT.format(dim=HYPERSPHERE_DIM, radius=radius)
    training_data, validation_data = HypersphereEmbeddedDynamicsDataset.load(
        dir_path=training_dir,
        n=HYPERSPHERE_DIM,
        radius=radius,
        device=device_cpu)

    training_dataloader = DataLoader(training_data,
                                     batch_size=batch_size, shuffle=True,
                                     num_workers=3, persistent_workers=True, prefetch_factor=8)
    validation_dataloader = DataLoader(validation_data,
                                       batch_size=batch_size, shuffle=True,
                                       num_workers=3, persistent_workers=True, prefetch_factor=8)

    # this is only for testing the baseline as if we're passing in extrinsic coordinates then we need to ensure that the input
    # dimension to the model is configured at the previous dimension
    input_dim = HYPERSPHERE_DIM if use_intrinsic else HYPERSPHERE_DIM + 1

    # if we're training the decoder-encoder with the extrinsic position/velocity directly then we need to use the ambient
    # dimension directly, if not then we assume we have a dimension of 2n which permits a smooth embedding via Whitney's
    # embedding theorem (from smooth manifold theory)
    learn_extrin_dim = (2 * HYPERSPHERE_DIM if mode != EmbeddingTrainingMode.TRAIN_EXTRINSIC_ENC_DEC_STREAM
                        else HYPERSPHERE_DIM + 1)

    # print(f"hypersphere dim: {HYPERSPHERE_DIM}")
    # print(f"input_dim: {input_dim}")
    # print(f"learn_extrin_dim: {learn_extrin_dim}")

    # print(f"use_intrinsic: {use_intrinsic}")

    # print(f"with prev state: {use_prev_state}")
    # print(f"use_intrinsic: {use_intrinsic}")

    # sets up the parameterized model

    model = EmbeddedDynamicsModel(
        intrin_dim=input_dim,
        learn_extrin_dim=learn_extrin_dim,
        dec_num_layers=dec_num_layers,
        dec_num_nodes_per_layer=dec_num_nodes,
        mlp_num_layers=mlp_num_layers,
        mlp_num_nodes_per_layer=mlp_num_nodes,
        enc_num_layers=enc_num_layers,
        enc_num_nodes_per_layer=enc_num_nodes,
        with_prev_state=use_prev_state,
        activation_fn=activation_fn,
    ).to(device_cuda)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)

    # pre-allocates space for the loss histories (epochs, num_batches)
    train_batch_pipeline_loss_hist = torch.zeros((TRAINING_EPOCHS, len(training_dataloader)), device=device_cpu)
    train_batch_dec_enc_loss_hist = torch.zeros((TRAINING_EPOCHS, len(training_dataloader)), device=device_cpu)
    train_batch_sparsity_loss_hist = torch.zeros((TRAINING_EPOCHS, len(training_dataloader)), device=device_cpu)
    valid_batch_pipeline_loss_hist = torch.zeros((TRAINING_EPOCHS, len(validation_dataloader)), device=device_cpu)

    # performs the actual training
    pbar = tqdm.tqdm(range(TRAINING_EPOCHS), desc="Training")
    for epoch in pbar:
        train_batch_pipeline_losses, train_batch_dec_enc_losses, train_batch_sparsity_losses = train_loop(
            dataloader=training_dataloader,
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            training_mode=mode,
            sparsity_penalty=penalty,
            use_intrinsic_pipeline_inputs=use_intrinsic,
        )
        # print(f"at end of training!")
        valid_batch_pipeline_losses = valid_loop(
            dataloader=validation_dataloader,
            model=model,
            loss_fn=loss_fn,
            use_intrinsic_pipeline_inputs=use_intrinsic,
        )

        train_batch_pipeline_loss_hist[epoch, :] = train_batch_pipeline_losses
        train_batch_dec_enc_loss_hist[epoch, :] = train_batch_dec_enc_losses
        train_batch_sparsity_loss_hist[epoch, :] = train_batch_sparsity_losses
        valid_batch_pipeline_loss_hist[epoch, :] = valid_batch_pipeline_losses

        pbar.set_postfix(
            avg_train_batch_loss=torch.mean(train_batch_pipeline_losses),
            avg_valid_batch_loss=torch.mean(valid_batch_pipeline_losses),
        )

        # print(f"done")
        # assert False

    # outputs this data for processing later
    train_batch_pipeline_loss_hist_numpy = train_batch_pipeline_loss_hist.numpy()
    train_batch_dec_enc_loss_hist_numpy = train_batch_dec_enc_loss_hist.numpy()
    train_batch_sparsity_loss_hist_numpy = train_batch_sparsity_loss_hist.numpy()
    valid_batch_pipeline_loss_hist_numpy = valid_batch_pipeline_loss_hist.numpy()

    dir_path = TRAINING_RESULTS_DIR / TRAINING_SUBDIR_FORMAT.format(dim=HYPERSPHERE_DIM, radius=radius)

    results_filename_prefix = (f"train_embed_results_mode_{mode}_dec_nl_{dec_num_layers}_dec_npl_{dec_num_nodes}"
                               f"_mlp_nl_{mlp_num_layers}_mlp_npl_{mlp_num_nodes}"
                               f"_enc_nl_{enc_num_layers}_enc_npl_{enc_num_nodes}"
                               f"_intrin_dim_{HYPERSPHERE_DIM}_learn_extrin_dim_{learn_extrin_dim}_"
                               f"_use_prev_state_{use_prev_state}"
                               f"_bs_{batch_size}_lr_{lr}_penalty_{penalty}"
                               f"_use_intrinsic_coords_{use_intrinsic}")
    results_data_name = results_filename_prefix + ".npz"
    results_model_name = results_filename_prefix + ".pt"

    torch.save(model.state_dict(), dir_path / results_model_name)
    np.savez(
        dir_path / results_data_name,

        # sphere params
        n=HYPERSPHERE_DIM,
        radius=radius,

        # training params
        batch_size=batch_size,
        lr=lr,
        sparsity_penalty=penalty,
        use_intrinsic_pipeline_inputs=use_intrinsic,
        mode=mode,

        # model params
        intrin_dim=HYPERSPHERE_DIM,
        learn_extrin_dim=learn_extrin_dim,
        dec_num_layers=dec_num_layers,
        dec_num_nodes_per_layer=dec_num_nodes,
        mlp_num_layers=mlp_num_layers,
        mlp_num_nodes_per_layer=mlp_num_nodes,
        enc_num_layers=enc_num_layers,
        enc_num_nodes_per_layer=enc_num_nodes,
        with_prev_state=use_prev_state,
        # activation_fn=activation_fn,

        # histories
        train_batch_pipeline_loss_hist_numpy=train_batch_pipeline_loss_hist_numpy,
        train_batch_dec_enc_loss_hist_numpy=train_batch_dec_enc_loss_hist_numpy,
        train_batch_sparsity_loss_hist_numpy=train_batch_sparsity_loss_hist_numpy,
        valid_batch_pipeline_loss_hist_numpy=valid_batch_pipeline_loss_hist_numpy,
    )

cfg: 5/30


Training: 100%|██████████| 1000/1000 [1:14:04<00:00,  4.44s/it, avg_train_batch_loss=tensor(7.1660e-05), avg_valid_batch_loss=tensor(7.4787e-06)]


cfg: 6/30


Training: 100%|██████████| 1000/1000 [1:06:17<00:00,  3.98s/it, avg_train_batch_loss=tensor(0.0265), avg_valid_batch_loss=tensor(0.0408)]


cfg: 7/30


...

cfg: 10/30


Training: 100%|██████████| 1000/1000 [1:06:24<00:00,  3.98s/it, avg_train_batch_loss=tensor(0.5036), avg_valid_batch_loss=tensor(0.5021)] 


cfg: 11/30


Training:  32%|███▏      | 324/1000 [20:07<42:00,  3.73s/it, avg_train_batch_loss=tensor(0.1388), avg_valid_batch_loss=tensor(0.1222)]   


KeyboardInterrupt: 